# rasterCube

In this tutorial we will discuss the Abstract Base Class `rasterCube`, which serves as the overarching blueprint in the raster data processing pipeline. The class can be visualized as the following UML diagram

```mermaid
classDiagram
    class dataCube {
        +logger
        +initialize_pipeline(recipe, logger) Tuple
    }

    class rasterEngine {
        <<Engine Component>>
        +affine_reproject()
        +compute_class_fraction()
        +_sanitize_spatial_geometry()
    }

    class rasterCube {
        <<abstract>>
        +GRID_REGISTRY : dict
        +generate_execution_plan(recipe, logger) DataFrame
        +parse_metadata(row, da) Tuple
        +process_cube(recipe, max_workers, logger) Dict
        +generate_raster_stac_item(paths, recipe, dataset_name, ...) Item
        +resolve_target_grid(spatial_cfg, logger)* str
        +get_resample_rule(variable_name)* str
        +get_auxiliary_dimensions(recipe)* Dict
    }

    class rasterDatasourceCube {
        <<Concrete Adapter>>
        +resolve_target_grid(spatial_cfg, logger) str
        +get_resample_rule(variable_name) str
        +get_auxiliary_dimensions(recipe) Dict
    }

    %% Relationships
    dataCube <|-- rasterCube : Inherits Orchestration
    rasterEngine <|-- rasterCube : Inherits GDAL/Warping Math
    rasterCube <|-- rasterDatasourceCube : Implements Abstract Methods
    ```

The Raster Cube ecosystem relies on a modular hierarchy that separates GDAL/xarray physical processing from orchestration and source-specific configuration.

*   **`dataCube` (The Operational Base):** Provides the foundational directory generation, logging telemetry, and hardware resource profiling (e.g., `initialize_pipeline`).
*   **`rasterEngine` (The Spatial Powerhouse):** A lower-level mixin containing the core GDAL C++ warping logic and geometric sanitation routines.
*   **`rasterCube` (The Abstract Orchestrator):** The master bridge that inherits from both `dataCube` and `rasterEngine`. It drives the universal out-of-core spatial processing loop (`process_cube()`), translates YAML recipes into a DataFrame queue (`generate_execution_plan()`), and compiles compliant STAC Items.
*   **`rasterDatasourceCube` (The Concrete Adapter):** The specialized child class (such as a connector for CLMS or CHELSA) that implements the abstract methods required to hook a specific dataset into the master spatial engine.

## Abstract Methods of the `rasterCube` Class

When creating your own `rasterDatasourceCube` by extending `rasterCube`, you are forced to define how your specific data is queried and mathematically processed by implementing three mandatory `@abstractmethod` contracts:

*   **`resolve_target_grid(spatial_cfg, logger)`:** Translates the user's YAML spatial configuration into a validated master grid key. This method must interpret the request and map it to a physically safe, mathematically supported grid framework present in the base class's `GRID_REGISTRY`.
*   **`get_resample_rule(variable_name)`:** Determines the appropriate GDAL spatial resampling algorithm (e.g., 'nearest', 'bilinear', 'average') for a given variable. This prevents data corruption by ensuring different physical and ecological variables receive strictly different mathematical treatments (such as keeping categorical land cover discrete).
*   **`get_auxiliary_dimensions(recipe)`:** Parses the YAML recipe to dynamically identify active auxiliary dimensions. It extracts dataset-specific parameters (like climate models or baseline periods) and maps them to exact column names in the GeoParquet catalog. These keys are dynamically appended to the Z-axis grouping logic during N-dimensional assembly.

### Implementing `resolve_target_grid`

The `resolve_target_grid` method enforces the spatial safety and physical limits of your data source. Its job is to intercept the user's YAML request (e.g., "1km EEA grid") and standardize it against the rigid `GRID_REGISTRY` mapping required by the GDAL warping engine. Crucially, this is where you write defensive logic to prevent users from artificially "upscaling" coarse data beyond its native limits.

#### Example 1: Enforcing Basic Atmospheric Caps (CHELSA)

In this CHELSA implementation, the engine enforces a strict `1km` cap for metric grids and a `30sec` cap for angular grids, preventing users from attempting to warp 1km climate data into high-resolution 10m pixels.

```python
def resolve_target_grid(
        self, 
        spatial_cfg: Dict[str, Any], 
        logger: Optional[logging.Logger] = None
    ) -> str:
        """
        Intersects target spatial configurations to enforce minimum atmospheric resolutions.
        Enforces 1km for metric grids (EEA, GEA) and 30sec for WGS84 degree grids.
        """
        log_execution(logger, "Resolving target grid base for CHELSA data cube generation...", logging.INFO)
        
        raw_grid = str(spatial_cfg.get('target_grid', 'EEA'))
        raw_res = str(spatial_cfg.get('target_resolution', '1km'))
        registry = getattr(self, 'GRID_REGISTRY', {})
        
        # 0. Early Exit: Exact match provided natively
        if raw_grid in registry:
            log_execution(logger, f"Successfully mapped to exact registry grid '{raw_grid}'.", logging.INFO)
            return raw_grid
            
        # 1. Base Grid Standardization via Dictionary Mapping
        grid_aliases = {
            "EEA": ("EEA", "1km"),
            "GEA": ("Global_EqualArea", "1km"),
            "GLOBAL_EQUALAREA": ("Global_EqualArea", "1km"),
            "GLOBAL_EQUAL_AREA": ("Global_EqualArea", "1km"),
            "WGS84": ("Global_WGS84", "30sec"),
            "GLOBAL_WGS84": ("Global_WGS84", "30sec")
        }
        
        # ... logic continues to reconstruct and return the grid key


#### Example 2: Dynamic Variable-Specific Caps (CLMS)

The CLMS implementation is more complex because its sub-datasets operate at different native resolutions. While standard land cover operates at 10m, the CLC+ instance data is capped at 100m. The adapter must check the active `variables` list and dynamically downgrade the requested grid if the user breaches the cap.

```python
def resolve_target_grid(
        self, 
        spatial_cfg: dict, 
        logger: logging.Logger = None,
        variables: list = None
    ) -> str:
        """
        Resolves the user's requested target grid by standardizing aliases, 
        and enforces physical resolution caps based on the native source datasets.
        """
        if variables is None:
            variables = []
            
        registry = getattr(self, 'GRID_REGISTRY', {})
        
        # [ ... Standardization and alias resolution logic ... ]
        
        target_res = registry[grid_key]["resolution"]
        crs = registry[grid_key]["crs"]
        
        is_degree = (crs == "EPSG:4326")
        
        # Default CLMS High Resolution Layer base resolution cap is 10 meters
        base_limit = 0.00008333333333333333 if is_degree else 10.0
        
        # The 'clcplus' (LULUCF-INSTANCE) product has a natively coarser base resolution of 100m.
        # If requested alongside HR layers, the entire output grid must be capped at 100m minimum.
        if "LULUCF-INSTANCE" in variables:
            base_limit = 0.0008333333333333333 if is_degree else 100.0
            
        # Enforce the cap: if requested grid is finer than base_limit, downgrade it
        if target_res < (base_limit * 0.99):
            if logger:
                logger.warning(
                    f"Requested grid '{grid_key}' ({target_res}) is finer than the "
                    f"source data cap ({base_limit}). Downgrading to the finest allowable limit."
                )
            
            # Filter for grids in the same CRS that satisfy the base limit cap
            valid_grids = {
                k: v for k, v in registry.items() 
                if v["crs"] == crs and v["resolution"] >= (base_limit * 0.99)
            }
            
            # Sort by resolution and pick the finest valid grid
            grid_key = sorted(valid_grids.items(), key=lambda item: item[1]["resolution"])[0][0]
            
        return grid_key

### Implementing `get_resample_rule`

The `get_resample_rule` method determines the appropriate GDAL spatial resampling algorithm for a given variable. This is critical for preventing data corruption during affine reprojection, ensuring that categorical classes are never interpolated and volume-based metrics are conserved.

#### Example 1: Categorical Routing (CLMS)

In the CLMS implementation, the adapter checks if the variable exists in the engine's internal `class_mapping`. If it is a strictly mapped categorical array (e.g., Crop Type), it triggers the specialized fractional coverage warping engine. Otherwise, it defaults to continuous interpolation.

```python
def get_resample_rule(self, variable_name: str) -> str:
    """
    Determines the appropriate spatial resampling algorithm for a variable.
    """
    # Strip potential temporal or string suffixes if the upstream pipeline appends them
    base_var = variable_name.split('_20')[0] if '_20' in variable_name else variable_name
    
    # Route strictly mapped categorical arrays (e.g., CTY, FTY) to the fractional engine
    if hasattr(self, 'class_mapping') and base_var in self.class_mapping:
        return "class_fraction"
        
    # Default to continuous bilinear resampling for confidence/density layers (e.g., Tree Cover Density)
    return "bilinear"
```

#### Example 2: Physical Safety Overrides (CHELSA)

The CHELSA implementation illustrates how to apply rigid physical constraints. While temperature might be resampled using bilinear interpolation, precipitation aggregates (like bioclimatic variables 12, 13, and 14) must conserve volume across spatial scales, forcing an override to the `average` algorithm.

```python
def get_resample_rule(self, variable_name: str) -> str:
    """
    Determines the appropriate GDAL spatial resampling algorithm for a variable.
    """
    var_str = str(variable_name)
    
    # Check the class dictionary for a registered rule, defaulting to bilinear
    rule = self.resample_rules.get(var_str, 'bilinear')
    
    # Implement safety override: composite precipitation models MUST conserve volume
    if 'bio' in var_str.lower():
        if var_str.lower() in ['bio12', 'bio13', 'bio14']:
            rule = 'average'
        else:
            rule = 'bilinear'
            
    return rule
```

### Implementing `get_auxiliary_dimensions`

The `get_auxiliary_dimensions` method parses the execution recipe to dynamically identify active, dataset-specific dimensions (like climate models, emission scenarios, or baseline periods). It extracts these parameters and maps them to the exact column names expected by the GeoParquet catalog.

#### Example 1: Direct Dictionary Extraction (CLMS)

For datasets with straightforward nesting, this method simply extracts the predefined auxiliary configuration block directly from the parsed YAML dictionary.

```python
def get_auxiliary_dimensions(self, recipe: Dict[str, Any]) -> Dict[str, List[Any]]:
    """
    Retrieves dataset-specific auxiliary dimension configurations from the recipe.
    """
    dataset_cfg = recipe.get('sources', {}).get(getattr(self, 'dataset_name', 'clms'), {})
    return dataset_cfg.get("auxiliary_dimensions", {})
```

#### Example 2: Multi-Level Parsing and Deduplication (CHELSA)

For complex datasets like CHELSA, climate projection parameters might be requested simultaneously across multiple structural levels (e.g., `climatologies` and `bioclim`). The adapter must scan these levels, deduplicate the requested models and scenarios using sets, and map the YAML keys to the specific Parquet catalog columns (e.g., mapping `ensembles` to `model`).

```python
def get_auxiliary_dimensions(self, recipe: Dict[str, Any]) -> Dict[str, List[Any]]:
    """
    Parses the execution recipe to identify and aggregate active CHELSA-specific 
    auxiliary dimensions across structural levels.
    """
    aux_dims = {}
    
    # Isolate the configuration block specific to the CHELSA dataset.
    chelsa_cfg = recipe.get('sources', {}).get(self.dataset_name, {})
    levels_cfg = chelsa_cfg.get('levels', {})
    
    # Initialize sets to automatically deduplicate requested dimensions
    active_ranges = set()
    active_ensembles = set()
    active_scenarios = set()
    
    # Iterate exclusively through the structural levels that utilize climate dimensions.
    for level in ['climatologies', 'bioclim']:
        settings = levels_cfg.get(level, {})
        
        # Only extract dimensions if the level itself is flagged for processing
        if settings.get('include', False):
            # Extract dictionary keys where the boolean value is True
            active_ranges.update(
                [tr for tr, state in settings.get('time_ranges', {}).items() if state]
            )
            active_ensembles.update(
                [ens for ens, state in settings.get('ensembles', {}).items() if state]
            )
            active_scenarios.update(
                [scen for scen, state in settings.get('scenarios', {}).items() if state]
            )
            
    # Map the extracted YAML configuration keys to the exact column names 
    # present in the underlying GeoParquet catalog.
    if active_ranges:
        aux_dims['time_range'] = list(active_ranges)
        
    if active_ensembles:
        # 'ensembles' in the user YAML corresponds to the 'model' column in the catalog
        aux_dims['model'] = list(active_ensembles)
        
    if active_scenarios:
        aux_dims['scenario'] = list(active_scenarios)
        
    return aux_dims
```

## Catalog Requirements and Schema Validation

To successfully route remote spatial assets through the pipeline, `rasterCube` relies on a highly structured execution manifest generated by `generate_execution_plan()`. This method operates by filtering an underlying spatial catalog (a Parquet file) based on the user's bounding box, temporal range, and variable requests.

For the engine to accept and process the catalog, the Parquet file must pass strict schema validation. The catalog must contain the following mandatory columns:

*   `geometry`
*   `datetime`
*   `level`
*   `variable`
*   `vsi_path`

If any of these columns are missing from the loaded Parquet file, the orchestrator will log a critical missing-column error and skip processing that catalog level entirely.

In [1]:
from bmc.cube.datasets.raster.chelsa import chelsaCube
import geopandas as gpd

cube = chelsaCube()
chelsa_catalog = gpd.read_parquet(cube.catalog)
chelsa_catalog

,geometry,level,variable,time_range,month,model,scenario,datetime,vsi_path,proj:epsg,nodata,asset_type
0,"POLYGON ((-180 -90, 180 -90, 180 90, -180 90, ...",annual,mymd,NaN,NaN,NaN,NaN,1980-01-01 00:00:00+00:00,/vsicurl/https://os.unil.cloud.switch.ch/chels...,4326,0.0,NaN
1,"POLYGON ((-180 -90, 180 -90, 180 90, -180 90, ...",annual,mymd,NaN,NaN,NaN,NaN,1981-01-01 00:00:00+00:00,/vsicurl/https://os.unil.cloud.switch.ch/chels...,4326,0.0,NaN
2,"POLYGON ((-180 -90, 180 -90, 180 90, -180 90, ...",annual,mymd,NaN,NaN,NaN,NaN,1982-01-01 00:00:00+00:00,/vsicurl/https://os.unil.cloud.switch.ch/chels...,4326,0.0,NaN
3,"POLYGON ((-180 -90, 180 -90, 180 90, -180 90, ...",annual,mymd,NaN,NaN,NaN,NaN,1983-01-01 00:00:00+00:00,/vsicurl/https://os.unil.cloud.switch.ch/chels...,4326,0.0,NaN
4,"POLYGON ((-180 -90, 180 -90, 180 90, -180 90, ...",annual,mymd,NaN,NaN,NaN,NaN,1984-01-01 00:00:00+00:00,/vsicurl/https://os.unil.cloud.switch.ch/chels...,4326,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
339220,"POLYGON ((-180 -90, 180 -90, 180 90, -180 90, ...",daily,we,NaN,NaN,NaN,NaN,2024-03-31 00:00:00+00:00,/vsicurl/https://os.unil.cloud.switch.ch/chels...,4326,65535.0,NaN
339221,"POLYGON ((-180 -90, 180 -90, 180 90, -180 90, ...",daily,we,NaN,NaN,NaN,NaN,2024-05-31 00:00:00+00:00,/vsicurl/https://os.unil.cloud.switch.ch/chels...,4326,65535.0,NaN
339222,"POLYGON ((-180 -90, 180 -90, 180 90, -180 90, ...",daily,we,NaN,NaN,NaN,NaN,2024-07-31 00:00:00+00:00,/vsicurl/https://os.unil.cloud.switch.ch/chels...,4326,65535.0,NaN
339223,"POLYGON ((-180 -90, 180 -90, 180 90, -180 90, ...",daily,we,NaN,NaN,NaN,NaN,2024-08-31 00:00:00+00:00,/vsicurl/https://os.unil.cloud.switch.ch/chels...,4326,65535.0,NaN


The GeoParquet catalog acts as the central nervous system for the spatial engine. Instead of brute-forcing API calls or hardcoding complex remote directory trees, the orchestrator queries this flat, indexed table to instantly locate the exact remote files (`vsi_path`) that intersect your recipe's spatial and temporal bounds.

Looking at the CHELSA catalog output, the schema can be broken down into three distinct tiers:

*   **The Mandatory Core:** These columns are strictly required by `generate_execution_plan()` to pass schema validation. 
    *   `geometry`: The native spatial footprint (represented as a Shapely Polygon) used for rapid bounding box intersections.
    *   `datetime`: The precise temporal anchor (ISO 8601) used for time-series filtering.
    *   `level`: The structural hierarchy or temporal resolution (e.g., `annual`, `daily`).
    *   `variable`: The specific physical metric (e.g., `mymd`, `we`).
    *   `vsi_path`: The GDAL-compatible Virtual File System URI pointing to the physical remote asset (e.g., `/vsicurl/https://...`).

*   **The Auxiliary Dimensions:** These are dataset-specific metadata columns. In the CHELSA example, columns like `model`, `scenario`, `time_range`, and `month` accommodate complex climate projections. When the engine calls your custom `get_auxiliary_dimensions()` method, it maps the user's YAML requests directly to these specific columns to filter the catalog down to the exact climate ensemble requested.

*   **The System Metadata:** Columns like `proj:epsg` (the native coordinate reference system) and `nodata` (the native masking value) keep the engine from flying blind. By storing these attributes physically in the catalog, the pipeline can pre-configure the GDAL C++ warping environments and allocate correct memory limits before a single byte of actual raster data is downloaded over the network.

### Accessing private and protected data repositories

The `rasterDatasourceCube` manages the authentication layer required to pull data from protected S3 buckets. Because different datasets rely on distinct cloud infrastructure, the engine relies on the `get_cloud_session()` and `resolve_provider()` methods to dynamically negotiate credentials.

When `process_cube()` initiates a network fetch, it inspects the first file path in the processing queue and passes it to `resolve_provider()`. This method analyzes the URI to determine which backend is hosting the data.

```python
def resolve_provider(self, paths: list) -> Optional[str]:
    """
    Determines the appropriate cloud provider ID from a list of URIs.
    """
    if not paths:
        return None
        
    first_path = str(paths[0])
    
    # 'eodata' is the standard bucket identifier for the Copernicus Data Space Ecosystem
    if "eodata" in first_path:
        return "cdse"
    # BII custom internal staging/production bucket
    elif "bii-bmd-workflow-engine-production" in first_path:
        return "private"
        
    return None

Once the provider is identified (e.g., "cdse" or "private"), the engine invokes `get_cloud_session()`. This function parses the respective JSON credential file and configures an active `AWSSession` to handle the authenticated connection, alongside a dictionary of GDAL HTTP environment variables (`env_kwargs`).

The handling of these credentials differs depending on whether the bucket is hosted on native AWS infrastructure or a custom third-party S3-compatible service:

#### Private Native AWS Repositories

For buckets hosted directly on Amazon Web Services (like the internal BII workflow bucket), GDAL's internal `/vsis3/` driver natively understands the authentication handshake. The engine only needs to supply standard Access Keys and the physical Region.

```python
if provider_id == "private" and self.private_credentials_path:
    # Load JSON credentials
    # ...
    
    # Native AWS Bucket: Omit endpoint_url, provide only standard region
    session = AWSSession(
        aws_access_key_id=priv.get("s3_access_key", priv.get("aws_access_key_id")),
        aws_secret_access_key=priv.get("s3_secret_key", priv.get("aws_secret_access_key")),
        region_name=priv.get("region_name", "eu-north-1")
    )
    return session, {}  # No explicit GDAL kwargs needed for native AWS infrastructure
```

#### Public Secured Repositories (Custom S3 Endpoints)

For buckets hosted on third-party infrastructure (like the Copernicus CDSE eodata bucket), the standard AWS handshake fails. GDAL must be explicitly told to route requests to a custom URL rather than `amazonaws.com`, and often requires specific overrides regarding Virtual Hosting styles and HTTPS mandates.

These overrides are passed back as `env_kwargs` so the orchestrator can inject them safely into a local `rasterio.Env` context block, preventing them from permanently polluting global system variables.

```python
if provider_id == "cdse" and self.cdse_credentials_path:
    # Load JSON credentials
    # ...
    
    # Custom Copernicus Bucket: Requires the endpoint protocol to be stripped 
    # so it can be injected directly into GDAL's internal C++ S3 driver configuration.
    raw_endpoint = cdse.get("s3_endpoint", "[https://eodata.dataspace.copernicus.eu](https://eodata.dataspace.copernicus.eu)")
    clean_endpoint = raw_endpoint.replace("https://", "").replace("http://", "").rstrip("/")
    
    session = AWSSession(
        aws_access_key_id=cdse.get("s3_access_key", cdse.get("aws_access_key_id")),
        aws_secret_access_key=cdse.get("s3_secret_key", cdse.get("aws_secret_access_key"))
    )
    env_kwargs = {
        "AWS_S3_ENDPOINT": clean_endpoint,
        "AWS_VIRTUAL_HOSTING": "FALSE",  # CDSE eodata uses path-style access, not virtual-hosted
        "AWS_HTTPS": "YES"
    }
    return session, env_kwargs
```

### Understanding the processing pipeline

```mermaid 
flowchart TD
    %% Define Nodes
    Start([Start process_cube])
    EndEmpty([Terminate Pipeline])
    End([End])

    %% Subgraph 1
    subgraph Preparation
        Init[1. Initialize Pipeline & Context]
        Plan[2. generate_execution_plan]
        CheckPlan{Is Plan Empty?}
        
        Init --> Plan --> CheckPlan
    end

    %% Subgraph 2
    subgraph Data Acquisition
        SpatialSetup[3. Resolve Target Grid & BBox Envelope]
        Fetch[4. parallel_fetch_rasters]
        
        SpatialSetup --> Fetch
    end

    %% Subgraph 3 (Contains Side-by-Side Workflows)
    subgraph Spatial Engine
        CheckTile{Asset Type == 'tile'?}

        %% Workflow A (Left)
        subgraph Mosaic Pipeline
            VRT[5a. Construct Virtual Raster]
            MetaMosaic[Parse Metadata]
            VRT --> MetaMosaic
        end

        %% Workflow B (Right)
        subgraph Timeseries Pipeline
            Raw[5b. Load Raw Fetched Array]
            MetaTimeseries[Parse Metadata]
            Raw --> MetaTimeseries
        end

        AlignWarp[5c. _align_warp_and_export]

        %% Branching creates the side-by-side layout
        CheckTile -- Yes --> VRT
        CheckTile -- No --> Raw

        %% Both pipelines converge here
        MetaMosaic --> AlignWarp
        MetaTimeseries --> AlignWarp
    end

    %% Subgraph 4
    subgraph Metadata
        STAC[6. generate_raster_stac_item]
    end

    %% Global Connections
    Start --> Init
    CheckPlan -- Yes --> EndEmpty
    CheckPlan -- No --> SpatialSetup
    Fetch --> CheckTile
    AlignWarp --> STAC
    STAC --> End
```